<div style="border-left:4px solid #34d399;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#34d399;">Understanding</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">How an English question becomes a schema, a symbol and an exact value.</div></div>

<div style="font:400 15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#3f3f46;">Everything on this page happens on this machine. Nothing has left yet, and by the end the question is ready to be sent with every real value removed.</div>

In [ ]:
# The code comes from GitHub. The repository is private, so a fresh clone needs a
# GITHUB_TOKEN secret (Add-ons -> Secrets).
#
# A version created through the API cannot read that secret - Kaggle answers 400
# no matter how the box is ticked in the editor, and the attachment cannot be
# declared in kernel-metadata.json either (Kaggle/kaggle-cli#582). Notebook 1's
# output carries the whole repository, so fall back to that copy. Only notebook 1
# has no input to fall back to, and only notebook 1 has to be saved from the
# browser rather than pushed.
import os, shutil, subprocess, sys
from pathlib import Path

ROOT = Path("/kaggle/working/nl2sql")


def clone_from_github() -> str:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", url, str(ROOT)], check=True)
    # git writes the clone URL into .git/config, token and all, and Kaggle saves
    # .git with the notebook output. Put the plain address back immediately.
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", "https://github.com/Kirazul/NL2SQL-demo.git"], check=True)
    return "a fresh clone"


def find_in_mounts(relative: str, max_depth: int = 7) -> "list[Path]":
    """Every path under /kaggle/input, so nothing has to guess how deep it is.

    Kaggle has mounted a notebook's output at /kaggle/input/<slug>/ and at
    /kaggle/input/notebooks/<owner>/<slug>/, and the repository sits a further
    level inside that. Each guess at the shape reported a perfectly good output
    as a missing database, so walk for it. Bounded, and never down into the two
    directories that hold the weights and the git objects, because an attached
    output is gigabytes.
    """
    root = Path("/kaggle/input")
    if not root.is_dir():
        return []
    hits = []
    for dirpath, dirnames, _ in os.walk(root):
        here = Path(dirpath)
        if len(here.relative_to(root).parts) >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if d not in {".git", "models", "wheels"}]
        candidate = here / relative
        if candidate.exists():
            hits.append(candidate)
    return sorted(hits)


def copy_from_setup() -> str:
    # Any pyproject.toml would match, so take the one with the package beside it.
    marker = next((m for m in find_in_mounts("pyproject.toml")
                   if (m.parent / "src" / "nl2sql").is_dir()), None)
    if marker is None:
        return ""
    # Everything except data/ and models/: those are the two gigabytes that get
    # read where they are mounted and are never worth copying.
    shutil.copytree(marker.parent, ROOT,
                    ignore=shutil.ignore_patterns("data", "models", ".git"))
    # The mount is read-only and copytree keeps the modes, but `pip install -e .`
    # writes an egg-info back into the tree.
    subprocess.run(["chmod", "-R", "u+w", str(ROOT)], check=True)
    return f"the copy in {marker.parent}"


def refresh_clone() -> str:
    """Pull new commits into a clone this session already has.

    Kaggle keeps /kaggle/working between runs of the same session, so the guard
    below used to leave whatever the first run cloned in place for hours. A cell
    then calls something added to the repository since, and the notebook reports
    ImportError for a name that is plainly there on main.
    """
    if not (ROOT / ".git").is_dir():
        return "the working directory (a copy, not a clone - left as it is)"
    from kaggle_secrets import UserSecretsClient
    try:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:  # noqa: BLE001
        print("not refreshed, GITHUB_TOKEN unreadable:", e)
        return "the working directory (not refreshed)"
    url = "https://github.com/Kirazul/NL2SQL-demo.git".replace("https://", f"https://{token}@")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", url, "main"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "FETCH_HEAD"], check=True)
    # `reset --hard` leaves untracked files alone, so the database, the index and
    # the weights that notebook 1 wrote here all survive.
    #
    # git records the URL it fetched from in .git/FETCH_HEAD, token and all, and
    # Kaggle saves .git with the notebook output. Remove it now, the same reason
    # the clone below puts the plain address back.
    (ROOT / ".git" / "FETCH_HEAD").unlink(missing_ok=True)
    head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
    return f"the working directory, refreshed to {head}"


if ROOT.exists():
    source = refresh_clone()
else:
    try:
        source = clone_from_github()
    except Exception as e:
        source = copy_from_setup()
        if not source:
            # Attached-but-empty and nothing-attached read identically from here,
            # and telling them apart is the whole difficulty, so name what is
            # mounted instead of guessing which one it is.
            mounted = sorted(p.name for p in Path("/kaggle/input").glob("*") if p.is_dir())
            where = ("the attached input(s) " + ", ".join(mounted) + " carry no "
                     "nl2sql/pyproject.toml") if mounted else "no input is attached"
            raise SystemExit(
                f"Nothing to run from: GITHUB_TOKEN could not be read "
                f"({type(e).__name__}: {e}), and {where}. Save this notebook from the "
                "browser, where the secret is readable - or attach a version of NL2SQL 1 "
                "Setup that ran to the end."
            ) from e
        print("GITHUB_TOKEN unreadable, falling back to notebook 1's output:", e)

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("code:", ROOT, "from", source)

In [ ]:
# pip writes to site-packages, which is not part of notebook 1's saved output, so
# every session installs again. Nearly all of it is already in the Kaggle image.
!pip install -q -e . 2>&1 | tail -2
print("dependencies ready")

In [ ]:
# Notebook 1 built the database, the index and the model weights and saved them
# with its output. Kaggle mounts that output read-only under /kaggle/input, and
# every one of the three is opened read-only here too - so point the settings at
# the mount rather than copying two gigabytes into the working directory.
#
# Where inside the mount they sit is not fixed - Kaggle has used
# /kaggle/input/<slug>/ and /kaggle/input/notebooks/<owner>/<slug>/ - so this
# walks for the database rather than matching a guessed shape. find_in_mounts
# comes from the first cell.
found = find_in_mounts("data/eicu.db")
if not found:
    # Printing the tree beats asserting a cause: the last two guesses at what
    # was wrong here were both wrong, and both would have been settled by this.
    print("nothing matched. What is actually under /kaggle/input:")
    root = Path("/kaggle/input")
    listed = 0
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > 3 or listed > 40:
            dirnames[:] = []
            continue
        print("   " * depth, Path(dirpath).name + "/", " ".join(sorted(filenames)[:6]))
        listed += 1
    raise SystemExit(
        "No data/eicu.db anywhere under /kaggle/input. Notebook 1's saved output is "
        "what carries it: check Input -> Add Input -> Your Work -> NL2SQL 1 Setup, and "
        "that the version pinned there is one that ran to the end."
    )
SETUP = found[0].parents[1]
print("setup output:", SETUP)

# Name a missing piece here rather than several cells later, from inside whichever
# library opens it first.
for path in (SETUP / "data/index.db", SETUP / "models/gliner2-base-v1"):
    if not path.exists():
        print("missing from notebook 1's output:", path)

# Set before nl2sql is imported anywhere: settings() is read once and cached. A
# subprocess started later - the API server in notebook 5 - inherits these too.
os.environ["DB_PATH"] = str(SETUP / "data" / "eicu.db")
os.environ["INDEX_PATH"] = str(SETUP / "data" / "index.db")
os.environ["GLINER_MODEL"] = str(SETUP / "models" / "gliner2-base-v1")
weights = sorted(SETUP.glob("models/*/*.gguf"))
if weights:
    os.environ["LOCAL_GGUF_PATH"] = str(weights[0])

for name in ("DB_PATH", "INDEX_PATH", "GLINER_MODEL", "LOCAL_GGUF_PATH"):
    print(f"  {name:<16} {os.environ.get(name, 'missing - the steps that need it will say so')}")

In [ ]:
# Keys live in Kaggle secrets, never in the notebook.
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
unreadable = []
for name in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "LANGSMITH_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        unreadable.append(name)

# All three failing at once is one cause, not three: a version created through the
# API cannot read a notebook secret however the editor shows it, and there is no
# field for the attachment in kernel-metadata.json (Kaggle/kaggle-cli#582). Say so
# once here rather than let every provider step fail separately further down.
if unreadable:
    print("could not read:", ", ".join(unreadable))
    if len(unreadable) == 3:
        print("A version created through the API cannot read notebook secrets. Save this")
        print("notebook from the browser to run the steps that call a provider.")

os.environ["LANGSMITH_TRACING"] = "1"
os.environ["LANGSMITH_PROJECT"] = "nl2sql"

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">1.</span> The database</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">What there is to ask about.</div></div>

In [ ]:
from nl2sql.db import schema

summary = schema.summary()
print(f"{summary['tables']} tables, {summary['columns']} columns, "
      f"{summary['rows']:,} rows, {summary['foreign_keys']} declared relationships")

print(schema.ddl({"patient", "medication"}))

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">2.</span> The index</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Two tiers. A column is indexed when its vocabulary is bounded, resolved on demand when it is not.</div></div>

In [ ]:
from nl2sql.db import values

stats = values.stats()
print("tier A (indexed) :", stats["tiers"].get("A"), "columns")
print("tier B (on demand):", stats["tiers"].get("B"), "columns")
print("values indexed   :", f"{stats['values_indexed']:,}", f"({stats['size_mb']} MB)")
print()
for ref, n in stats["top"][:6]:
    print(f"  {ref:<44} {n:>6}")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">The cost of the index is bounded by the number of columns, not by the number of rows. That is what makes it transfer to a database far larger than this one.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">3.</span> Finding a value</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The analyst writes a word; the database holds something longer.</div></div>

In [ ]:
for mention in ["aspirin", "asspirin", "sepsis", "female"]:
    found = values.search(mention, limit=1)
    if found:
        best = found[0]
        print(f"{mention:<12} -> {best.value[:46]:<48} {best.ref:<28} {best.score:.2f}")
    else:
        print(f"{mention:<12} -> nothing")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">4.</span> Naming a column</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Not every word is content. Some name a column, and the two compete.</div></div>

In [ ]:
from nl2sql.db import catalog

for mention in ["diagnosis names", "administration routes", "aspirin"]:
    match = catalog.best(mention)
    if match:
        print(f"{mention:<24} -> {match.ref:<34} {match.score:.2f}  ({match.why})")
    else:
        print(f"{mention:<24} -> no column matches it, so it can only be a value")

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">A real value scores low against every column, and a column name scores low against every value. That gap is what lets the pipeline tell them apart instead of guessing.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">5.</span> Reading the question</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">The three previous steps, run together, with every decision recorded.</div></div>

In [ ]:
from nl2sql.core import trace
from nl2sql.nlp.understand import understand

trace.configure()
with trace.record('How many patients over 65 received aspirin?') as run:
    u = understand('How many patients over 65 received aspirin?')

for step in run.steps:
    print(f"  {step.label:<40} {step.ms:>7.0f} ms  {step.summary}")

In [ ]:
print("tables:", sorted(u.tables))
print()
for r in u.resolutions:
    kind = r.kind.upper()
    value = f"-> {r.value}" if r.value else ""
    print(f"  {r.mention:<16} {kind:<10} {str(r.column):<34} {value}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">6.</span> Hiding the values</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Each value becomes a symbol. The mapping stays here.</div></div>

In [ ]:
from nl2sql.privacy.mask import mask

masked = mask(u)
print("before:", u.question)
print("after :", masked.question)
print()
for symbol, value in masked.mapping.items():
    print(f"  {symbol} = {value!r}   from {masked.columns.get(symbol, 'the analyst')}")

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">7.</span> The message that would be sent</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Assembled, then checked part by part before any connection is opened.</div></div>

In [ ]:
from nl2sql.core import prompt
from nl2sql.privacy import gate

built = prompt.hybrid(u, masked)
for verdict in gate.verdicts(built.segments):
    mark = "ok " if verdict["allowed"] else "NO "
    print(f"  {mark}{verdict['origin']:<10} {verdict['checked_by']:<30} {verdict['preview'][:52]}")

In [ ]:
print(built.messages[-1]["content"])

<div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;">Not one real value appears above. The provider is given the shape of the database and a sentence with holes in it.</div>

<div style="margin:22px 0 10px;"><div style="font:600 16px/1.3 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#18181b;"><span style="color:#34d399;">8.</span> What it refuses</div><div style="font:400 14px/1.6 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#52525b;margin-top:4px;">Two questions that cannot be answered, and are stopped here rather than half-answered.</div></div>

In [ ]:
from nl2sql.privacy.mask import UnmaskableQuestion, UnresolvableValue

for question in ["Did Mr. Bensalah receive insulin?", "How many patients received asparatan?"]:
    try:
        mask(understand(question))
        print(f"{question}\n  -> sent\n")
    except (UnmaskableQuestion, UnresolvableValue) as e:
        print(f"{question}\n  -> {e}\n")